# 03. CNN — 이미지를 이미지답게 다루기

노트북 01에서 이미지를 다룰 때마다 `reshape`로 **1차원으로 펼쳤다(flatten)**.
그리고 그때마다 말했다 — "이 과정에서 **픽셀의 위치 정보가 사라진다**"고.

28×28 이미지를 784개 숫자로 펼치면, 모델은 **어떤 픽셀이 어떤 픽셀 옆에 있었는지** 모른다.
사람 얼굴에서 눈이 코 위에 있다는 것, 숫자 6의 아래쪽에 동그라미가 있다는 것 —
이런 **공간 구조**를 DNN은 처음부터 못 본다. (노트북 02에서 얼굴 분류가 처참히 실패한 이유 중 하나다.)

**CNN(합성곱 신경망, Convolutional Neural Network)** 은 바로 이 문제를 푼다.
이미지를 펼치지 않고 **2차원 그대로** 받아서, 위치 관계를 유지한 채 특징을 뽑아낸다.

이 노트북에서 다루는 것:
1. **합성곱(Conv2D)과 풀링(MaxPooling)** 이 무엇인지
2. Fashion-MNIST를 CNN으로 분류 (노트북 01의 DNN과 비교)
3. **같은 CNN 구조로 MNIST vs Fashion-MNIST** — 데이터 난이도의 차이


## CNN의 두 가지 핵심 연산

### 1. 합성곱 (Convolution) — `Conv2D`

작은 **필터(filter, 또는 커널)** 를 이미지 위에서 **미끄러뜨리며(sliding)** 훑는다.
필터는 예를 들어 5×5 크기의 작은 가중치 판이다. 이 판을 이미지의 각 위치에 대고,
그 자리의 픽셀들과 곱해 더한 값 하나를 만든다. 이걸 이미지 전체에 반복하면 새로운 2차원 지도가 나온다.

- 한 필터는 하나의 **특징(feature)** 을 감지한다 — 세로선, 가로선, 곡선, 특정 질감 등.
- `Conv2D(filters=32, ...)` = "**서로 다른 특징 32개**를 감지하는 필터 32장을 학습하라".
- 핵심 이점: **필터가 이미지 어디를 훑든 같은 가중치를 쓴다.** 그래서
  ① 위치 정보가 보존되고, ② 왼쪽 위의 곡선이든 오른쪽 아래의 곡선이든 **같은 필터로 잡아낸다**(위치 불변성).
  ③ Dense와 달리 파라미터가 훨씬 적다 (5×5 필터 = 25개 가중치로 이미지 전체를 훑는다).

`padding='same'` : 필터가 가장자리를 훑을 때 이미지가 줄어들지 않도록 테두리를 0으로 채운다.
결과 지도의 크기가 입력과 같아진다(same).

### 2. 풀링 (Pooling) — `MaxPooling2D`

합성곱 결과를 **작게 요약**한다. `MaxPooling2D(pool_size=(2,2))` 는
2×2 영역마다 **가장 큰 값 하나만** 남긴다 → 가로세로가 **절반**으로 줄어든다 (28×28 → 14×14).

- **왜?** ① 계산량을 줄이고, ② "이 근처에 이 특징이 있다"는 대략적 위치만 남겨
  약간의 위치 변화에 둔감해진다(작은 이동에 강해짐).

### CNN의 전형적 흐름

```
[Conv2D → MaxPooling] 를 여러 번 반복  →  Flatten  →  Dense(분류)
   (특징을 뽑고 요약)                    (그때서야 펼침)  (판단)
```

**펼치기(Flatten)는 맨 마지막에 딱 한 번** 한다 — 특징을 다 뽑고 난 뒤, 분류 직전에.
DNN처럼 처음부터 펼치지 않는 게 핵심이다.


## 1부. Fashion-MNIST를 CNN으로

### 1-1. 데이터 불러오기와 전처리

노트북 01과 결정적으로 다른 점: **flatten하지 않는다.**

- DNN에서는 `reshape(60000, -1)` 로 784차원으로 펼쳤다.
- CNN은 2차원 형태를 유지하되, **채널 축**만 하나 추가한다: `(28, 28)` → `(28, 28, 1)`.
  - 흑백 이미지는 채널 1개(밝기), 컬러라면 3개(RGB). `Conv2D`는 이 채널 축을 요구한다.
  - `x_train[:, :, :, np.newaxis]` 가 맨 뒤에 크기 1인 축을 붙인다.
- 스케일링(`/255`)과 원-핫은 01과 동일.


In [1]:
from tensorflow.keras.datasets import fashion_mnist
import numpy as np
from tensorflow.keras.utils import to_categorical

(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()

# flatten 대신 채널 축(마지막)만 추가: (N,28,28) → (N,28,28,1)
x_train_3d = x_train[:, :, :, np.newaxis] / 255
x_test_3d = x_test[:, :, :, np.newaxis] / 255
y_train_oh = to_categorical(y_train)
y_test_oh = to_categorical(y_test)

print("입력 shape:", x_train_3d.shape)   # (60000, 28, 28, 1)
print("정답 shape:", y_train_oh.shape)   # (60000, 10)


29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 1us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
입력 shape: (60000, 28, 28, 1)
정답 shape: (60000, 10)


### 1-2. CNN 모델 만들기

```
Input(28,28,1)
  → Conv2D(32, 5×5, same, relu) → MaxPooling(2×2)   # 28×28 → 14×14
  → Conv2D(32, 5×5, same, relu) → MaxPooling(2×2)   # 14×14 → 7×7
  → Conv2D(32, 5×5, same, relu) → MaxPooling(2×2)   # 7×7 → 3×3
  → Flatten()                                        # 3×3×32 = 288
  → Dense(10, softmax)
```

- Conv/Pooling을 3번 반복하며 **특징을 뽑고 크기를 줄인다** (28→14→7→3).
- `padding='same'` 덕에 Conv 자체로는 크기가 안 줄고, **줄이는 건 오직 Pooling**이다.
- 마지막에 `Flatten`으로 3×3×32=288을 펼쳐서 `Dense(10)`으로 분류.
- 활성화·손실·옵티마이저는 노트북 01과 같다 (relu / softmax / categorical_crossentropy / adam).

**주목할 점: 파라미터 수.** 이 CNN은 약 5만 개다.
노트북 01의 DNN(같은 Fashion-MNIST)은 21만 개였다. **CNN이 4배 이상 적은 파라미터로** 더 나은 일을 한다.


In [2]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Conv2D, MaxPooling2D, Flatten

model = Sequential()
model.add(Input(shape=(28, 28, 1)))
model.add(Conv2D(filters=32, kernel_size=(5, 5), padding='same', activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Conv2D(filters=32, kernel_size=(5, 5), padding='same', activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Conv2D(filters=32, kernel_size=(5, 5), padding='same', activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Flatten())
model.add(Dense(10, activation='softmax'))
model.summary()

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 28, 28, 32)     │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 14, 14, 32)     │        25,632 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 7, 7, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 7, 7, 32)       │        25,632 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 3, 3, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 288)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 10)             │         2,890 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 54,986 (214.79 KB)

 Trainable params: 54,986 (214.79 KB)

 Non-trainable params: 0 (0.00 B)

### 1-3. 학습

Fashion-MNIST는 어려운 편이라 5에폭만 돌려도 경향을 볼 수 있다.
(원본 수업에서는 콜백 없이 5에폭을 돌렸다. 여기서도 그대로 두고 결과를 관찰한다.)


In [3]:
hist = model.fit(x_train_3d, y_train_oh,
                 validation_data=(x_test_3d, y_test_oh),
                 epochs=5)


Epoch 1/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 135s 71ms/step - accuracy: 0.8318 - loss: 0.4684 - val_accuracy: 0.8729 - val_loss: 0.3532
Epoch 2/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 131s 70ms/step - accuracy: 0.8876 - loss: 0.3102 - val_accuracy: 0.8867 - val_loss: 0.3143
Epoch 3/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 143s 71ms/step - accuracy: 0.9031 - loss: 0.2655 - val_accuracy: 0.9030 - val_loss: 0.2781
Epoch 4/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 133s 71ms/step - accuracy: 0.9128 - loss: 0.2381 - val_accuracy: 0.9033 - val_loss: 0.2657
Epoch 5/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 145s 72ms/step - accuracy: 0.9210 - loss: 0.2140 - val_accuracy: 0.8966 - val_loss: 0.2884


### 1-4. DNN vs CNN 비교 ★

노트북 01의 DNN과 이번 CNN을 같은 Fashion-MNIST로 나란히 놓으면:

| | 노트북 01 DNN | 이번 CNN |
|---|---|---|
| 입력 처리 | flatten (784) — **위치 정보 버림** | 2차원 유지 (28×28×1) |
| 파라미터 | **218,058개** | **54,986개** (약 1/4) |
| 학습 에폭 | 50 (콜백 없이) | **5** |
| Fashion-MNIST 검증 정확도 | 최고 약 0.896 (12에폭 부근) | **0.897** (5에폭) |

숫자를 읽으면 CNN의 강점이 분명해진다:

- **파라미터가 1/4인데 정확도는 비슷하거나 살짝 더 높다.** DNN이 50에폭에 걸쳐 겨우 도달한 0.89대를,
  CNN은 **단 5에폭에** 4배 적은 파라미터로 달성했다.
- 이유는 **필터 재사용**이다. DNN은 784개 입력마다 별도의 가중치를 두지만(그래서 21만 개),
  CNN은 5×5 필터 하나(25개 가중치)를 이미지 전체에 미끄러뜨려 재사용한다. 낭비가 없다.
- 게다가 CNN은 **위치 정보를 버리지 않는다.** 이게 진짜 이미지(얼굴, 사진)로 갈수록 결정적 차이가 된다.

> **핵심: CNN은 이미지를 이미지답게 다룬다.** 펼쳐서 위치를 버리는 대신,
> 공간 구조를 유지한 채 특징을 뽑는다. 적은 파라미터, 빠른 수렴, 더 나은 일반화.

---

## 정리

| 개념 | 핵심 |
|---|---|
| **Conv2D** | 작은 필터를 이미지 위에 미끄러뜨려 특징을 뽑는다. 필터 하나 = 특징 하나, 전체에 재사용 |
| **padding='same'** | Conv로는 크기가 안 줄게 테두리를 0으로 채운다. 크기는 Pooling이 줄인다 |
| **MaxPooling2D** | 2×2마다 최댓값만 남겨 절반으로 요약. 계산량↓, 작은 위치 변화에 둔감 |
| **Flatten** | 특징을 다 뽑은 **맨 마지막**에 한 번만. DNN처럼 처음부터 펼치지 않는다 |
| **CNN vs DNN** | 1/4 파라미터, 1/10 에폭으로 비슷한 정확도. 위치 정보 보존 |

**다음 노트북(04)** 에서는 CNN을 실제 사진(개 vs 고양이)에 적용하면서,
`EarlyStopping`과 `ModelCheckpoint`로 **최적 모델을 자동 저장**하는 법을 본다.
그리고 노트북 02에서 봤던 "데이터 부족" 문제가 CNN에서도 어떻게 나타나는지 확인한다.